# Phase 3: Customer Feature Engineering

In this phase, we transform transactional order-level data into 
customer-level features suitable for segmentation and retention modeling.

Objectives:
- Aggregate behavioral metrics at customer level
- Construct RFM (Recency, Frequency, Monetary) features
- Generate additional revenue and engagement metrics
- Prepare final modeling dataset

In [ ]:
# ============================================
# 1. Imports
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

In [ ]:
# ============================================
# 2. Datetime Handling
# ============================================
master_df = pd.read_csv("../data/processed/master_dataset.csv")

master_df['order_purchase_timestamp'] = pd.to_datetime(
    master_df['order_purchase_timestamp']
)

## 1. RFM Feature Construction

RFM analysis is a standard customer segmentation technique:

- Recency: How recently a customer made a purchase
- Frequency: How often the customer purchases
- Monetary: How much revenue the customer generates

These features form the foundation of segmentation strategy.

In [ ]:
# ============================================
# 3. Defining Snapshot Date
# ============================================

snapshot_date = master_df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

print("Snapshot Date:", snapshot_date)
#This represents the "current" date for recency calculation

In [ ]:
# ============================================
# 4. Creating RFM Metrics
# ============================================

rfm = (
    master_df
    .groupby('customer_id')
    .agg({
        'order_purchase_timestamp': lambda x: (snapshot_date - x.max()).days,
        'order_id': 'nunique',
        'payment_value': 'sum'
    })
    .reset_index()
)

rfm.columns = [
    'customer_id',
    'recency_days',
    'frequency',
    'monetary'
]

rfm.head()


### RFM Feature Meaning

- Lower recency → More recently active customers
- Higher frequency → Loyal / repeat customers
- Higher monetary → High-value customers

In [ ]:
# ============================================
# 5. RFM Distribution
# ============================================

rfm[['recency_days', 'frequency', 'monetary']].describe()

In [ ]:
# ============================================
# RFM Distribution Overview (Combined View)
# ============================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Recency
axes[0].hist(rfm['recency_days'], bins=40)
axes[0].set_title("Recency Distribution")
axes[0].set_xlabel("Days Since Last Purchase")

# Frequency
axes[1].hist(rfm['frequency'], bins=30)
axes[1].set_title("Frequency Distribution")
axes[1].set_xlabel("Number of Orders")

# Monetary
axes[2].hist(rfm['monetary'], bins=40)
axes[2].set_title("Monetary Distribution")
axes[2].set_xlabel("Total Spending")

plt.tight_layout()
plt.show()